In [6]:
import datetime
import pandas as pd
import numpy as np

print("Loading AI Chip Market Data...")
df = pd.read_csv('ai_chip_market.csv')

print("\n--- Data Health Check ---")
print(df.info())

df['launch_date'] = pd.to_datetime(df['launch_date'])
print(f"\nDate conversion successful. Earliest launch: {df['launch_date'].min().date()}, Latest: {df['launch_date'].max().date()}")

Loading AI Chip Market Data...

--- Data Health Check ---
<class 'pandas.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   year                       120 non-null    int64  
 1   chip_name                  120 non-null    str    
 2   vendor                     120 non-null    str    
 3   launch_date                120 non-null    str    
 4   memory_gb                  120 non-null    int64  
 5   fp16_tflops                120 non-null    int64  
 6   tdp_watts                  120 non-null    int64  
 7   estimated_shipments_units  120 non-null    int64  
 8   estimated_asp_usd          120 non-null    int64  
 9   estimated_revenue_usd_m    120 non-null    float64
 10  description                120 non-null    str    
dtypes: float64(1), int64(6), str(4)
memory usage: 16.7 KB
None

Date conversion successful. Earliest launch: 2017-05-10

In [7]:
current_year = datetime.datetime.now().year

df['chip_age_years'] = current_year - df['launch_date'].dt.year

conditions = [
    (df['chip_age_years'] >= 4) & (df['estimated_revenue_usd_m'] < 100),
    (df['chip_age_years'] >= 3) & (df['estimated_revenue_usd_m'] < 500)
]
choices = ['High EOL Risk', 'Medium Risk']

df['obsolescence_status'] = np.select(conditions, choices, default='Healthy')

print("\n--- Obsolescence Distribution ---")
print(df['obsolescence_status'].value_counts())


--- Obsolescence Distribution ---
obsolescence_status
Healthy          65
High EOL Risk    40
Medium Risk      15
Name: count, dtype: int64


In [3]:
vendor_share = df.groupby('vendor').agg(
    total_revenue_m=('estimated_revenue_usd_m', 'sum'),
    total_units_shipped=('estimated_shipments_units', 'sum')
).reset_index().sort_values(by='total_revenue_m', ascending=False)

print("\n--- Vendor Market Share ---")
print(vendor_share.head())

top_chips = df.groupby(['chip_name', 'vendor'])['estimated_revenue_usd_m'].sum().reset_index()
top_chips = top_chips.sort_values(by='estimated_revenue_usd_m', ascending=False).head(5)

print("\n--- Top 5 Chips by Revenue ---")
print(top_chips)


--- Vendor Market Share ---
   vendor  total_revenue_m  total_units_shipped
8  NVIDIA         254064.5              8964751
5  Google          23365.8              2735594
0     AMD          19166.6              1044482
7  Huawei          11707.5               754463
1     AWS           7973.8               531586

--- Top 5 Chips by Revenue ---
         chip_name  vendor  estimated_revenue_usd_m
24     NVIDIA H100  NVIDIA                 181554.7
25     NVIDIA H200  NVIDIA                  37646.3
12  Google TPU v5e  Google                  23282.1
1       AMD MI300X     AMD                  17030.1
26     NVIDIA L40S  NVIDIA                  12875.4


In [4]:
output_filename = 'ai_chip_market_cleaned.csv'
df.to_csv(output_filename, index=False)
print(f"\nPipeline complete. Cleaned data exported to {output_filename}")


Pipeline complete. Cleaned data exported to ai_chip_market_cleaned.csv
